### Initialization

In [1]:
import pandas as pd
from datasets import load_dataset
from torchao.quantization import (
    Int8DynamicActivationInt8WeightConfig,
    Int8WeightOnlyConfig,
)
from transformers import AutoModelForTokenClassification
import torch

from sequence_tagging_benchmark.taggers import (
    NltkHMMTagger,
    HmmlearnTagger,
    CRFTagger,
    BiLSTMTagger,
    TransformerTagger,
    QuantizedBiLSTMTagger,
    QuantizedTransformerTagger,
    ONNXTransformerTagger,
)

/Users/nikita/anaconda3/envs/nlp_case_study/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0315 11:57:46.885000 85582 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
NUM_THREADS = 8

torch.set_num_threads(NUM_THREADS)

### Data loading

In [3]:
dataset = load_dataset("universal_dependencies", "en_ewt")

train_data = dataset["train"]
test_data = dataset["test"]

upos_feature = train_data.features["upos"].feature
int2str = upos_feature.int2str

In [4]:
raw_train_tokens = [row["tokens"] for row in train_data]
raw_train_tags = [[int2str(tag_id) for tag_id in row["upos"]] for row in train_data]

raw_test_tokens = [row["tokens"] for row in test_data]
raw_test_tags = [[int2str(tag_id) for tag_id in row["upos"]] for row in test_data]

### Benchmarking

In [5]:
# I noticed that most of transformer models have much diverse tag set then specified in upos
# Though, their tag set could be mapped into original upos
# by removing prefix and suffix that provide additional linguistic information
def clean_transformer_upos_tag(raw_tag: str) -> str:
    if raw_tag.startswith("B-") or raw_tag.startswith("I-"):
        raw_tag = raw_tag[2:]
    base_tag = raw_tag.split("|")[0]
    base_tag = base_tag.split("+")[0]
    return base_tag

In [15]:
TAG_TYPE = "upos"
BLSTM_MODEL_NAME = "flair/upos-multi"
TRANSFORMER_MODEL_NAME = "KoichiYasuoka/roberta-base-english-upos"

transformer_id2label = {
    id: clean_transformer_upos_tag(base_label)
    for id, base_label in AutoModelForTokenClassification.from_pretrained(
        TRANSFORMER_MODEL_NAME
    ).config.id2label.items()
}

nltk_hmm = NltkHMMTagger()
nltk_hmm.train(raw_train_tokens, raw_train_tags)

hmmlearn_hmm = HmmlearnTagger()
hmmlearn_hmm.train(raw_train_tokens, raw_train_tags)

crf = CRFTagger()
crf.train(raw_train_tokens, raw_train_tags)

bilstm = BiLSTMTagger(model_name=BLSTM_MODEL_NAME, tag_type=TAG_TYPE)
bilstm.train(raw_train_tokens, raw_train_tags)

dynamic_quantized_bilstm = QuantizedBiLSTMTagger(
    model_name=BLSTM_MODEL_NAME,
    tag_type=TAG_TYPE,
    quantize_config=Int8DynamicActivationInt8WeightConfig(),
)
dynamic_quantized_bilstm.train(raw_train_tokens, raw_train_tags)

weight_only_quantized_bilstm = QuantizedBiLSTMTagger(
    model_name=BLSTM_MODEL_NAME,
    tag_type=TAG_TYPE,
    quantize_config=Int8WeightOnlyConfig(),
)
weight_only_quantized_bilstm.train(raw_train_tokens, raw_train_tags)

transformer = TransformerTagger(
    model_name=TRANSFORMER_MODEL_NAME,
    id2label=transformer_id2label,
)
transformer.train(raw_train_tokens, raw_train_tags)

dynamic_quantized_transformer = QuantizedTransformerTagger(
    model_name=TRANSFORMER_MODEL_NAME,
    id2label=transformer_id2label,
    quantize_config=Int8DynamicActivationInt8WeightConfig(),
)
dynamic_quantized_transformer.train(raw_train_tokens, raw_train_tags)

weight_only_quantized_transformer = QuantizedTransformerTagger(
    model_name=TRANSFORMER_MODEL_NAME,
    id2label=transformer_id2label,
    quantize_config=Int8WeightOnlyConfig(),
)
weight_only_quantized_transformer.train(raw_train_tokens, raw_train_tags)

onnx_transformer = ONNXTransformerTagger(
    model_name=TRANSFORMER_MODEL_NAME,
    id2label=transformer_id2label,
)
onnx_transformer.train(raw_train_tokens, raw_train_tags)

2026-03-15 12:03:04,649 SequenceTagger predicts: Dictionary with 17 tags: NOUN, PUNCT, ADP, VERB, ADJ, DET, PROPN, ADV, PRON, AUX, CCONJ, NUM, SCONJ, PART, X, SYM, INTJ
2026-03-15 12:03:05,601 SequenceTagger predicts: Dictionary with 17 tags: NOUN, PUNCT, ADP, VERB, ADJ, DET, PROPN, ADV, PRON, AUX, CCONJ, NUM, SCONJ, PART, X, SYM, INTJ
2026-03-15 12:03:06,159 SequenceTagger predicts: Dictionary with 17 tags: NOUN, PUNCT, ADP, VERB, ADJ, DET, PROPN, ADV, PRON, AUX, CCONJ, NUM, SCONJ, PART, X, SYM, INTJ


/Users/nikita/anaconda3/envs/nlp_case_study/lib/python3.12/site-packages/torchao/quantization/quant_api.py:925: UserWarning: Config Deprecation: version 1 of Int8WeightOnlyConfig is deprecated and will no longer be supported in a future release, please use version 2, see https://github.com/pytorch/ao/issues/2752 for more details
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
/Users/nikita/anaconda3/envs/nlp_case_study/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask


In [18]:
# Evaluate all models
results = []

# Traditional models
results.append(
    {
        "name": "nltk_hmm",
        **nltk_hmm.evaluate(raw_test_tokens, raw_test_tags, batch_size=1),
    }
)
results.append(
    {
        "name": "hmmlearn_hmm",
        **hmmlearn_hmm.evaluate(raw_test_tokens, raw_test_tags, batch_size=1),
    }
)
results.append(
    {"name": "crf", **crf.evaluate(raw_test_tokens, raw_test_tags, batch_size=1)}
)

# BiLSTM variants
results.append(
    {
        "name": "bilstm-batch-1",
        **bilstm.evaluate(raw_test_tokens, raw_test_tags, batch_size=1),
    }
)
results.append(
    {
        "name": "bilstm-batch-128",
        **bilstm.evaluate(raw_test_tokens, raw_test_tags, batch_size=128),
    }
)
results.append(
    {
        "name": "bilstm-batch-128-smart-batching",
        **bilstm.evaluate(
            raw_test_tokens, raw_test_tags, batch_size=128, use_smart_batching=True
        ),
    }
)
results.append(
    {
        "name": "dynamic-quantized-bilstm-batch-128",
        **dynamic_quantized_bilstm.evaluate(
            raw_test_tokens, raw_test_tags, batch_size=128
        ),
    }
)
results.append(
    {
        "name": "weight-only-quantized-bilstm-batch-128",
        **weight_only_quantized_bilstm.evaluate(
            raw_test_tokens, raw_test_tags, batch_size=128
        ),
    }
)

# Transformer variants
results.append(
    {
        "name": "transformer-batch-1",
        **transformer.evaluate(raw_test_tokens, raw_test_tags, batch_size=1),
    }
)
results.append(
    {
        "name": "transformer-batch-128",
        **transformer.evaluate(raw_test_tokens, raw_test_tags, batch_size=128),
    }
)
results.append(
    {
        "name": "transformer-batch-128-smart-batching",
        **transformer.evaluate(
            raw_test_tokens, raw_test_tags, batch_size=128, use_smart_batching=True
        ),
    }
)
results.append(
    {
        "name": "dynamic-quantized-transformer-batch-128",
        **dynamic_quantized_transformer.evaluate(
            raw_test_tokens, raw_test_tags, batch_size=128
        ),
    }
)
results.append(
    {
        "name": "weight-only-quantized-transformer-batch-128",
        **weight_only_quantized_transformer.evaluate(
            raw_test_tokens, raw_test_tags, batch_size=128
        ),
    }
)
results.append(
    {
        "name": "onnx-transformer-batch-128",
        **onnx_transformer.evaluate(raw_test_tokens, raw_test_tags, batch_size=128),
    }
)
results = pd.DataFrame(results)

Benchmarking: 100%|██████████| 17/17 [01:23<00:00,  4.91s/it]


In [19]:
results

,name,micro_f1,macro_f1,p50_latency_ms,p90_latency_ms,p99_latency_ms,latency_variance_ms,throughput_seq_per_sec,inference_memory_delta_mb,final_memory_retained_mb
0,nltk_hmm,0.869873,0.803750,0.495167,1.361958,2.774398,0.339075,1517.083273,2.250000,2.250000
1,hmmlearn_hmm,0.874867,0.793628,0.132709,0.155867,0.243160,0.000432,7223.063080,0.015625,0.015625
2,crf,0.932125,0.912911,0.024917,0.068650,0.132723,0.000789,30401.975949,0.000000,0.000000
3,bilstm-batch-1,0.918086,0.849703,222.714208,623.654758,1191.617444,63771.116137,3.405419,3.437500,-559.343750
4,bilstm-batch-128,0.918086,0.849703,27.815006,46.627455,48.742120,86.475541,34.227067,709.437500,709.437500
5,bilstm-batch-128-smart-batching,0.918086,0.849703,19.898752,39.387171,100.753940,220.301723,44.505828,96.171875,96.171875
6,dynamic-quantized-bilstm-batch-128,0.918007,0.849301,31.476029,50.355302,52.726018,94.244674,30.361846,23.593750,23.593750
7,weight-only-quantized-bilstm-batch-128,0.918007,0.848844,28.510217,45.530058,49.189650,78.057602,33.442412,5.421875,4.953125
8,transformer-batch-1,0.942821,0.840820,39.539666,43.087466,48.579895,735.098996,25.130083,0.000000,-899.203125
9,transformer-batch-128,0.942821,0.840820,8.702597,15.302604,41.976373,69.908695,91.363136,773.468750,684.015625


In [ ]:
results.to_csv("../artifacts/pos_tagging_results.csv", index=False)